# Agentomics - a PydanticAI-powered autonomous ML Agent

Requirements:
- Internet connection
- Git
- Docker installed and running
- Conda installed
- Jupyter kernel support for your selected Python environment
- About 25 GB free disk space
- An OpenRouter API key for the live run

## Before Running Cells

If VS Code says the selected Python environment requires `ipykernel`, install it in the environment selected as the notebook kernel before running this notebook.

If your terminal prompt starts with `(base)`, use Conda:

```bash
conda install -n base ipykernel -y
```

If you are using a virtual environment instead, activate it and run:

```bash
python -m pip install ipykernel -U
```

Avoid installing into `/bin/python3` on managed Linux distributions. If you see `externally-managed-environment`, switch to Conda or a virtual environment.

After installing, re-select the notebook kernel in VS Code. This setup must happen before notebook cells can execute.

## 1. Clone Agentomics-ML

Run this from your workshop directory. The cell clones Agentomics-ML into `agentomics-ml` the first time. If that folder already contains Agentomics-ML, the cell reuses it.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess

workshop_root = Path.cwd().resolve()
repo_dir = workshop_root / "agentomics-ml"

if repo_dir.exists():
    required_files = [repo_dir / "run.sh", repo_dir / "scripts" / "download_example_dataset.sh"]
    if not all(path.exists() for path in required_files):
        raise FileExistsError(
            f"{repo_dir} already exists, but it does not look like Agentomics-ML. "
            "Move or rename it, then rerun this cell."
        )
    print(f"Using existing repository: {repo_dir}")
else:
    subprocess.run(
        ["git", "clone", "https://github.com/BioGeMT/agentomics-ml.git", str(repo_dir)],
        check=True,
    )
    print(f"Repository cloned to: {repo_dir}")

os.chdir(repo_dir)
print(f"Current directory: {Path.cwd()}")

Cloning into '/home/nucleotaid/Downloads/agentomics-ml'...


Repository cloned to: /home/nucleotaid/Downloads/agentomics-ml
Current directory: /home/nucleotaid/Downloads/agentomics-ml


## 2. Confirm Prerequisites

These checks confirm the machine is ready before the long cells start.

In [3]:
def run(command):
    return subprocess.run(command, text=True, capture_output=True, check=False)

for tool in ["git", "docker", "conda"]:
    path = shutil.which(tool)
    print(f"{tool}: {path}")
    if path is None:
        raise RuntimeError(f"Missing required tool: {tool}")

docker_status = run(["docker", "info"])
if docker_status.returncode != 0:
    raise RuntimeError("Docker is not running or is not accessible.")
print("Docker is running.")

free_space_gb = shutil.disk_usage(repo_dir).free / 1024**3
print(f"Free space: {free_space_gb:.1f} GB")
if free_space_gb < 25:
    raise RuntimeError("At least 25 GB free disk space is recommended for this workshop.")


git: /usr/bin/git
docker: /usr/bin/docker
conda: /home/nucleotaid/miniconda3/bin/conda
Docker is running.
Free space: 1448.8 GB


## 3. Configure API Key

Paste the workshop OpenRouter key below. The cell writes `.env` in the cloned repository.

Do not commit or share the generated `.env` file.

In [ ]:
openrouter_api_key = ""  # Paste the workshop key here, for example: sk-or-v1-...

if not openrouter_api_key.startswith("sk"):
    raise ValueError("Paste a valid OpenRouter API key before continuing.")

env_path = repo_dir / ".env"
env_path.write_text(f"OPENROUTER_API_KEY={openrouter_api_key}\n")
print(f"Wrote API key configuration to {env_path}")


## 4. Download Example Dataset

The workshop dataset is `AGO2_CLASH_Hejret2023`.

In [18]:
dataset_name = "AGO2_CLASH_Hejret2023"

subprocess.run(
    ["./scripts/download_example_dataset.sh", "--dataset", dataset_name],
    check=True,
)

dataset_dir = repo_dir / "datasets" / dataset_name
print(f"Dataset ready: {dataset_dir}")


2 channel Terms of Service accepted

Remove all packages in environment /home/nucleotaid/miniconda3/envs/agentomics-datasets:


## Package Plan ##

  environment location: /home/nucleotaid/miniconda3/envs/agentomics-datasets


The following packages will be REMOVED:

  _openmp_mutex-4.5-20_gnu
  bzip2-1.0.8-hda65f42_9
  ca-certificates-2026.4.22-hbd8a1cb_0
  ld_impl_linux-64-2.45.1-default_hbd61a6d_102
  libblas-3.11.0-7_h4a7cf45_openblas
  libcblas-3.11.0-7_h0358290_openblas
  libexpat-2.8.0-hecca717_0
  libffi-3.5.2-h3435931_0
  libgcc-15.2.0-he0feb66_19
  libgcc-ng-15.2.0-h69a702a_19
  libgfortran-15.2.0-h69a702a_19
  libgfortran5-15.2.0-h68bc16d_19
  libgomp-15.2.0-he0feb66_19
  liblapack-3.11.0-7_h47877c9_openblas
  liblzma-5.8.3-hb03c661_0
  liblzma-devel-5.8.3-hb03c661_0
  libnsl-2.0.1-hb9d3cd8_1
  libopenblas-0.3.33-pthreads_h94d23a6_0
  libsqlite-3.53.1-h0c1763c_0
  libstdcxx-15.2.0-h934c35e_19
  libuuid-2.42-h5347b49_0
  libzlib-1.3.2-h25fd6f3_2
  ncurses-6.6-hdb14827_0
  num

## 5. Preview Dataset

Run these cells to inspect the dataset description, the train/test files, and the supervised learning task: labeled examples in, trained classifier out.

In [19]:
import csv


dataset_name = "AGO2_CLASH_Hejret2023"
dataset_dir = repo_dir / "datasets" / dataset_name
train_path = dataset_dir / "train.csv"
test_path = dataset_dir / "test.csv"
description_path = dataset_dir / "dataset_description.md"


def csv_shape(path):
    with path.open(newline="") as file:
        reader = csv.reader(file)
        header = next(reader)
        row_count = sum(1 for _ in reader)
    return row_count, len(header)


def csv_columns(path):
    with path.open(newline="") as file:
        return next(csv.reader(file))


def csv_first_row(path):
    with path.open(newline="") as file:
        reader = csv.DictReader(file)
        return next(reader)


def print_tree(root, max_depth=2):
    root = Path(root)
    print(root)

    def walk(directory, prefix="", depth=0):
        if depth >= max_depth:
            return
        entries = sorted(directory.iterdir(), key=lambda path: (path.is_file(), path.name.lower()))
        for index, path in enumerate(entries):
            connector = "└── " if index == len(entries) - 1 else "├── "
            print(prefix + connector + path.name)
            if path.is_dir():
                extension = "    " if index == len(entries) - 1 else "│   "
                walk(path, prefix + extension, depth + 1)

    walk(root)


print("Dataset description:\n")
print(description_path.read_text())

print("Train shape:", csv_shape(train_path))
print("Test shape:", csv_shape(test_path))


Dataset description:

The AGO2 Hejret2023 dataset was adapted from [miRBench: novel benchmark datasets for microRNA binding site prediction that mitigate against prevalent microRNA Frequency Class Bias]. This dataset contains microRNA sequences and their corresponding binding sites, as identified via a CLASH (crosslinking, ligation, and sequencing of hybrids) experiment. There are two sequences in this dataset: gene and noncodingRNA. The gene sequences are 50nt fragments including a target site of the noncodingRNA. We expect that the targeting occurs via partial complementarity of the two sequences. Samples with label==1 are target sites retrieved from the CLASH experiment. For each of these positive samples, a negative sample (label==0) is created by matching the same noncodingRNA sequence with a randomly selected gene sequence.
Train shape: (8193, 11)
Test shape: (965, 11)


In [5]:
print_tree(dataset_dir, max_depth=1)


/home/nucleotaid/Downloads/agentomics-ml/datasets/AGO2_CLASH_Hejret2023
├── dataset_config.json
├── dataset_description.md
├── test.csv
└── train.csv


In [6]:
print("Columns:")
print(csv_columns(train_path))

print("\nFirst train row:")
csv_first_row(train_path)


Columns:
['gene', 'noncodingRNA', 'noncodingRNA_name', 'noncodingRNA_fam', 'feature', 'target', 'chr', 'start', 'end', 'strand', 'gene_cluster_ID']

First train row:


{'gene': 'CCCAGGGTGTTTCATGCTGAGGTAGTAGGATGAATAAAGGCAAATATGCA',
 'noncodingRNA': 'CTATACAATCTACTGTCTTTC',
 'noncodingRNA_name': 'hsa-let-7a-3p',
 'noncodingRNA_fam': 'let-7',
 'feature': '',
 'target': '1',
 'chr': '13',
 'start': '103634174',
 'end': '103634223',
 'strand': '-',
 'gene_cluster_ID': '355'}

## 6. Run Agentomics

This workshop run uses CPU-only execution and no foundation models. Keeping the run small makes it practical to follow during the session.

Run configuration:
- Provider: OpenRouter
- Model: `gpt-5.1-codex-max`
- Iterations: 2
- Verbosity: `summary`
- Hardware mode: CPU only
- Foundation models: disabled by not passing `--foundation-models-type`

The first run on a fresh machine may spend several minutes pulling Docker images.

In [ ]:
dataset_name = "AGO2_CLASH_Hejret2023"
!./run.sh --provider openrouter --dataset {dataset_name} --model gpt-5.1-codex-max --iterations 2 --val-metric AUROC --cpu-only --verbosity summary


## 7. Find Run Output

In [5]:
outputs_dir = repo_dir / "outputs"
agent_dirs = sorted(
    [path for path in outputs_dir.glob("*") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not agent_dirs:
    raise FileNotFoundError("No Agentomics output directories found.")

agent_dir = agent_dirs[0]
print("Latest output:", agent_dir)


Latest output: /home/nucleotaid/Downloads/agentomics-ml/outputs/hedonistic_joseph_list


In [8]:
print_tree(agent_dir, max_depth=2)


/home/nucleotaid/Downloads/agentomics-ml/outputs/hedonistic_joseph_list
├── .git
│   ├── branches
│   ├── hooks
│   ├── info
│   ├── logs
│   ├── objects
│   ├── refs
│   ├── COMMIT_EDITMSG
│   ├── config
│   ├── description
│   ├── HEAD
│   └── index
├── best_iteration_snapshot
│   ├── .conda
│   ├── data_exploration
│   ├── data_representation
│   ├── data_split
│   ├── iteration_plan
│   ├── model_architecture
│   ├── model_inference
│   ├── model_training
│   ├── prediction_exploration
│   ├── runtime_info
│   ├── validation_evaluation
│   ├── environment.yml
│   ├── eval_predictions_test.csv
│   └── test_metrics.json
├── extras
├── fallbacks
├── reports
│   ├── markdown
│   └── pdf
├── run
│   ├── iteration_0
│   ├── iteration_1
│   └── shared
├── .gitignore
└── README.md


## 8. Explore Iteration Files

Connect the workflow figure to the folders inside each `run/iteration_*` directory.

In [9]:
run_dir = agent_dir / "run"
iteration_dirs = sorted(run_dir.glob("iteration_*"))
print(f"Found {len(iteration_dirs)} iteration folders")
print("Run directory:", run_dir)


Found 2 iteration folders
Run directory: /home/nucleotaid/Downloads/agentomics-ml/outputs/hedonistic_joseph_list/run


In [10]:
print_tree(run_dir, max_depth=2)


/home/nucleotaid/Downloads/agentomics-ml/outputs/hedonistic_joseph_list/run
├── iteration_0
│   ├── data_exploration
│   ├── data_representation
│   ├── data_split
│   ├── iteration_plan
│   ├── model_architecture
│   ├── model_inference
│   ├── model_training
│   ├── prediction_exploration
│   ├── runtime_info
│   └── validation_evaluation
├── iteration_1
│   ├── data_exploration
│   ├── data_representation
│   ├── data_split
│   ├── iteration_plan
│   ├── model_architecture
│   ├── model_inference
│   ├── model_training
│   ├── prediction_exploration
│   ├── runtime_info
│   └── validation_evaluation
└── shared
    ├── .conda
    ├── datasets
    ├── splits
    ├── config.json
    └── environment.yml


In [11]:
import json

output_json_paths = sorted(run_dir.glob("iteration_*/*/output.json"))
print(f"Found {len(output_json_paths)} step output.json files")

sample_output_path = output_json_paths[0]
print("Sample:", sample_output_path.relative_to(agent_dir))
with sample_output_path.open() as file:
    sample_output = json.load(file)
sample_output


Found 18 step output.json files
Sample: run/iteration_0/data_exploration/output.json


{'step_id': 'data_exploration',
 'model_type': 'DataExplorationOutput',
 'payload': {'files_created': [],
  'data_description': "Dataset loaded from /workspace/run/shared/datasets/AGO2_CLASH_Hejret2023/train.csv with 8,193 rows and 12 columns. Key columns: gene (50 nt fragments), noncodingRNA (17–26 nt, mean ~22 nt), metadata (noncodingRNA_name, noncodingRNA_fam, feature, chr, start, end, strand, gene_cluster_ID, id) and numeric_label (target). No duplicate rows. Missing values only in 'feature' (756 rows); all other fields complete.",
  'feature_analysis': "Labels are nearly balanced: 0=4,109, 1=4,084. gene sequences have fixed length 50 (std=0), noncodingRNA lengths range 17–26 (mean ~22.07, std ~0.93). No missing values in sequence or label columns. Metadata mostly complete except 'feature' missing in ~9.2% of rows.",
  'domain_insights': "Sequence-pair classification with partial complementarity expectation. Fixed-length gene (50 nt) simplifies k-mer windowing; variable-length nonc

## 9. Inspect Best Iteration Snapshot

`best_iteration_snapshot/` contains the selected model artifacts and inference code.

In [12]:
best_snapshot = agent_dir / "best_iteration_snapshot"
print("Best iteration snapshot:", best_snapshot)


Best iteration snapshot: /home/nucleotaid/Downloads/agentomics-ml/outputs/hedonistic_joseph_list/best_iteration_snapshot


In [13]:
print_tree(best_snapshot, max_depth=3)


/home/nucleotaid/Downloads/agentomics-ml/outputs/hedonistic_joseph_list/best_iteration_snapshot
├── .conda
│   └── envs
│       └── hedonistic_joseph_list_env
├── data_exploration
│   └── output.json
├── data_representation
│   ├── representation_char1mer
│   │   ├── id_train.joblib
│   │   ├── id_val.joblib
│   │   ├── rep_info.json
│   │   ├── vec_gene.joblib
│   │   ├── vec_nc.joblib
│   │   ├── X_train.npz
│   │   ├── X_val.npz
│   │   ├── y_train.joblib
│   │   └── y_val.joblib
│   ├── build_representation.py
│   └── output.json
├── data_split
│   └── output.json
├── iteration_plan
│   └── output.json
├── model_architecture
│   └── output.json
├── model_inference
│   ├── inference.py
│   └── output.json
├── model_training
│   ├── helpers
│   │   ├── __pycache__
│   │   ├── __init__.py
│   │   └── training_reporter.py
│   ├── training_artifacts
│   │   ├── meta.json
│   │   ├── model.joblib
│   │   ├── val_predictions.csv
│   │   ├── vec_gene.joblib
│   │   └── vec_nc.joblib
│   ├─

## 10. Review Reports

Review only the reports for the selected best iteration. The PDF is embedded below when it exists.

In [14]:
from IPython.display import IFrame, Markdown, display

metadata_path = best_snapshot / "runtime_info" / "iteration_metadata.json"
if not metadata_path.exists():
    raise FileNotFoundError("Best iteration metadata was not found. The run may not have completed successfully.")

with metadata_path.open() as file:
    best_iteration_metadata = json.load(file)

best_iteration = best_iteration_metadata["iteration"]
best_markdown_report = agent_dir / "reports" / "markdown" / f"run_report_iter_{best_iteration}.md"
best_pdf_report = agent_dir / "reports" / "pdf" / f"iteration_{best_iteration}.pdf"

print(f"Best iteration: {best_iteration}")
print("Markdown report:", best_markdown_report.relative_to(agent_dir))
print("PDF report:", best_pdf_report.relative_to(agent_dir))

if not best_markdown_report.exists():
    raise FileNotFoundError(f"Missing markdown report: {best_markdown_report}")
if not best_pdf_report.exists():
    raise FileNotFoundError(f"Missing PDF report: {best_pdf_report}")


Best iteration: 1
Markdown report: reports/markdown/run_report_iter_1.md
PDF report: reports/pdf/iteration_1.pdf


In [15]:
markdown_preview = best_markdown_report.read_text()
display(Markdown(markdown_preview))


# Run Report - Iteration 1

**Agent ID:** `hedonistic_joseph_list`  
**Dataset:** `AGO2_CLASH_Hejret2023`  
**Model:** `gpt-5.1-codex-max`  
**Task:** `classification`  
**Validation metric:** `AUROC`  
**Generated:** 2026-05-14 10:54:15

---

## Iteration Plan

**Data Exploration Instructions:**

> Skip this step; reuse understanding from iteration_0 exploration (balanced labels, fixed 50nt gene, variable-length 17-26nt noncodingRNA).

**Data Split Instructions:**

Skip this step; reuse split version 0 from iteration_0 (train.csv and validation.csv already provided).

**Data Representation Instructions:**

> Implement a new basic representation not used before: character 1-mer (mononucleotide) counts separately for gene and noncodingRNA using CountVectorizer(analyzer='char', ngram_range=(1,1), lowercase=False). Fit two vectorizers on train split only (one per column), transform train/val, then hstack into sparse matrices. Save vectorizers and transformed matrices plus a small rep_info.json/joblib with shapes and vocab sizes.

**Model Architecture Instructions:**

> Use a simple classical model not tried before: Multinomial Naive Bayes. Plan to tune alpha over [0.1, 0.5, 1.0] using validation AUROC. No class weights. Use partial_fit not needed; just fit per alpha.

**Model Training Instructions:**

> Write a new train.py that loads train/val CSVs from the current split, uses the saved 1-mer vectorizers to transform sequences (or fits inside if needed), trains MultinomialNB for each alpha in [0.1,0.5,1.0], evaluates on validation using AUROC (also log_loss, accuracy, AUPRC for reference), selects best alpha by AUROC, refits on full training data with that alpha, and saves model plus vectorizers in a training_artifacts folder. Include a meta.json with chosen alpha and metrics.

**Model Inference Instructions:**

> Create inference.py that loads the saved MultinomialNB model and both 1-mer vectorizers, transforms input gene and noncodingRNA columns, outputs CSV with columns: id, prediction (0/1), probability_0, probability_1. Use decision_threshold=0.5 on proba_1 for prediction.

**Prediction Exploration Instructions:**

> Evaluate on validation split: save val_preds.csv with id, true label, pred_proba; compute AUROC, AUPRC, accuracy, F1, log_loss, MCC; save ROC and PR curve CSVs and a brief txt summary. Highlight best alpha found.

**Other Instructions:**

_None_

## Data Exploration

**Data Description:**

Exploration skipped

**Feature Analysis:**

Exploration skipped

**Domain Insights:**

Exploration skipped

## Data Split

**Train Path:**

/workspace/run/shared/splits/split_0/train.csv

**Val Path:**

/workspace/run/shared/splits/split_0/validation.csv

**Splitting Strategy:**

> Stratified random 80/20 split on numeric_label with random_state=42. All columns preserved; no shuffling beyond the stratified split. Saved to train.csv and validation.csv in current step directory.

**Split Changed:**

False

**Split Version:**

0

## Data Representation

**Representation:**

> Character 1-mer counts for gene and noncodingRNA separately using CountVectorizer(analyzer='char', ngram_range=(1,1), lowercase=False); fitted on train split only. Transform train/validation, then hstack into sparse matrices (CSR). Saved artifacts in representation_char1mer/: X_train.npz, X_val.npz, y_train.joblib, y_val.joblib, id_train.joblib, id_val.joblib, vec_gene.joblib, vec_nc.joblib, rep_info.json (includes shapes and vocab sizes). Train shape: (6554, 8), Val shape: (1639, 8).

**Files created:**

- `build_representation.py`
- `y_val.joblib`
- `X_val.npz`
- `id_train.joblib`
- `vec_nc.joblib`
- `id_val.joblib`
- `vec_gene.joblib`
- `X_train.npz`
- `rep_info.json`
- `y_train.joblib`

## Model Architecture

**Architecture:**

> Multinomial Naive Bayes on character 1-mer count features for gene and noncodingRNA (separate CountVectorizers, hstacked)

**Hyperparameters:**

Tune alpha over {0.1, 0.5, 1.0}; no class weights; fit_prior=True; decision threshold 0.5 on proba_1

## Model Training

**Path To Train File:**

/workspace/run/current_iteration/model_training/train.py

**Path To Model File:**

/workspace/run/current_iteration/model_training/training_artifacts/model.joblib

**Path To Artifacts Dir:**

/workspace/run/current_iteration/model_training/training_artifacts

**Training Summary:**

> Implemented character 1-mer CountVectorizer features for gene and noncodingRNA, tuned MultinomialNB alphas (0.1,0.5,1.0) using validation AUROC via single-pass fits, selected best alpha, refit on full training data, and saved model, vectorizers, meta, and validation predictions.

**Files created:**

- `train.py`
- `meta.json`
- `val_predictions.csv`
- `vec_nc.joblib`
- `vec_gene.joblib`
- `model.joblib`

## Model Inference

**Path To Inference File:**

/workspace/run/current_iteration/model_inference/inference.py

**Inference Summary:**

> Inference script loads MultinomialNB plus char 1-mer CountVectorizers for gene and noncodingRNA, transforms input columns, computes predict_proba, applies 0.5 threshold for class prediction, and saves CSV with id, prediction, probability_0, probability_1. Artifacts dir defaults to training_artifacts but is configurable.

**Files created:**

- `inference.py`

## Prediction Exploration

**Statistics:**

Validation AUROC: 0.474; AUPRC: 0.522; Accuracy: 0.476; F1: 0.453; Log-loss: 0.695; MCC: -0.049

**Insights:**

> The character 1-mer MultinomialNB performs poorly, with AUROC below random and slight negative MCC, indicating the model is miscalibrated and not discriminative. Predicted positive rate by label shows bias toward predicting positives (positive_rate≈0.50 for label 0 vs. 0.45 for label 1), and high-confidence bins are rare—most probabilities cluster near 0.5, suggesting the model cannot separate classes on this representation.

**Files created:**

- `analyze_predictions.py`
- `pr_curve.csv`
- `summary.txt`
- `val_preds.csv`
- `metrics.json`
- `correct_cases.csv`
- `wrong_cases.csv`
- `roc_curve.csv`

## Validation Evaluation

**Metrics:**

> {
>   "train/ACC": 0.5056454073848031,
>   "train/AUPRC": 0.5404523486535489,
>   "train/AUROC": 0.5052604946124872,
>   "train/F1": 0.5048419945146603,
>   "train/LOG_LOSS": 0.6932883674507093,
>   "train/MCC": 0.0110876956127427,
>   "validation/ACC": 0.4758999389871873,
>   "validation/AUPRC": 0.5223939847790529,
>   "validation/AUROC": 0.47386438426740757,
>   "validation/F1": 0.47499624305057586,
>   "validation/LOG_LOSS": 0.6949922661216129,
>   "validation/MCC": -0.04859965843729246
> }

**Is New Best:**

True

## Metrics

- **validation/ACC**: 0.4758999389871873
- **validation/AUPRC**: 0.5223939847790529
- **validation/AUROC**: 0.47386438426740757
- **validation/F1**: 0.47499624305057586
- **validation/LOG_LOSS**: 0.6949922661216129
- **validation/MCC**: -0.04859965843729246
- **train/ACC**: 0.5056454073848031
- **train/AUPRC**: 0.5404523486535489
- **train/AUROC**: 0.5052604946124872
- **train/F1**: 0.5048419945146603
- **train/LOG_LOSS**: 0.6932883674507093
- **train/MCC**: 0.0110876956127427

## Test Metrics

- **ACC**: 0.49430051813471504
- **AUPRC**: 0.5561035813905256
- **AUROC**: 0.49195357833655706
- **F1**: 0.4942087264353594
- **LOG_LOSS**: 0.693939490102244
- **MCC**: -0.010047208163259016


## 11. Run Inference on a Small Train Subset

Create a small unlabeled subset from `train.csv`, run the selected inference pipeline on that subset, and inspect the predictions.

In [25]:
label_column = "target"
sample_size = 1000
inference_input = repo_dir / "new_samples.csv"
predictions_path = repo_dir / "new_predictions.csv"

with train_path.open(newline="") as input_file, inference_input.open("w", newline="") as output_file:
    reader = csv.DictReader(input_file)
    fieldnames = [column for column in reader.fieldnames if column != label_column]
    writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    writer.writeheader()
    for row_number, row in enumerate(reader):
        if row_number >= sample_size:
            break
        writer.writerow({column: row[column] for column in fieldnames})

print(f"Created {sample_size} unlabeled rows from train.csv")
print("Inference input:", inference_input)


Created 1000 unlabeled rows from train.csv
Inference input: /home/nucleotaid/Downloads/agentomics-ml/new_samples.csv


In [26]:
!./scripts/inference.sh --cpu-only --agent-dir {agent_dir} --input {inference_input} --output {predictions_path}


Using code path: best_iteration_snapshot
Running in CPU-only mode
[Warning] Input CSV has no 'id' column. Sequential IDs (0..N-1) added in a temporary file, used for running inference on. If you need specific IDs, include an 'id' column in the input csv
Running inference in Docker...
Inference done


In [28]:
print("Predictions file:", predictions_path)
print("Predictions shape:", csv_shape(predictions_path))
!head -5 {predictions_path}


Predictions file: /home/nucleotaid/Downloads/agentomics-ml/new_predictions.csv
Predictions shape: (1000, 4)
id,prediction,probability_0,probability_1
0,1,0.48542450895683875,0.514575491043155
1,0,0.5328111603940424,0.46718883960596236
2,0,0.5196611259242463,0.4803388740757615
3,0,0.5295645211902398,0.4704354788097487
